# BERT Fine-tuning — `career_position` Annotation

Fine-tune a multilingual BERT model (`bert-base-multilingual-cased`) to classify
job descriptions into CoREx career position codes.

**Task:** annotation / `career_position`  
**Input:** `job_description_label` (silver standard text)  
**Output:** career position code (e.g. `401 = politics`)  

**Requirements:** `pip install corex_eval[bert]`

## 1. Imports & config

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

from corex_eval import evaluate, load_inputs, load_training_data

from dotenv import load_dotenv, dotenv_values


# --- Hyperparameters ---
MODEL_NAME  = "bert-base-multilingual-cased"
MAX_LENGTH  = 128
EPOCHS      = 5
BATCH_SIZE  = 32
LR          = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
SEED        = 42

OUTPUT_DIR  = Path("checkpoints")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


## 2. Load training data

In [2]:
train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position"],
)
train_df = (
    train_df
    .dropna(subset=["job_description_label", "career_position"])
    .pipe(lambda df: df[df["job_description_label"].str.strip() != ""])
    .reset_index(drop=True)
)
train_df["input_text"] = (
    train_df["job_description_label"] + " | " + train_df["workplace_label"].fillna("")
)
print(f"{len(train_df)} training rows")
train_df[["job_description_label", "workplace_label", "career_position"]].head()

FileNotFoundError: Gold standard file not found: data/gold/corex_gold.csv
Set the COREX_DATA_DIR environment variable to the folder containing your data, or pass an explicit path to load_gold().
Expected structure:
  $COREX_DATA_DIR/
  └── gold/
      └── corex_gold.csv

## 3. Build label mapping

In [ ]:
unique_labels = sorted(train_df["career_position"].unique())
label2id = {lbl: i for i, lbl in enumerate(unique_labels)}
id2label = {i: lbl for i, lbl in enumerate(unique_labels)}
num_labels = len(unique_labels)

print(f"{num_labels} unique career_position codes:")
for code in unique_labels:
    print(f"  {code}")

## 4. Tokenise

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_encodings = tokenizer(
    train_df["input_text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
)
train_labels = [label2id[lbl] for lbl in train_df["career_position"]]


class SpellDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = SpellDataset(train_encodings, train_labels)
print(f"Dataset size: {len(train_dataset)}")

## 5. Fine-tune

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": float((preds == labels).mean())}

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    seed=SEED,
    save_strategy="epoch",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

## 6. Predict on test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")

test_df["input_text"] = (
    test_df["job_description_label"] + " | " + test_df["workplace_label"].fillna("")
)

test_encodings = tokenizer(
    test_df["input_text"].tolist(),
    truncation=True,
    padding="max_length",
    max_length=MAX_LENGTH,
    return_tensors="pt",
)

model.eval()
model.to(device)

with torch.no_grad():
    outputs = model(**{k: v.to(device) for k, v in test_encodings.items()})

pred_ids = outputs.logits.argmax(dim=-1).cpu().numpy()
pred_codes = [id2label[int(i)] for i in pred_ids]

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = pred_codes
predictions_df.head()

## 7. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    submit=True,
    experiment_path="/home/tom/projects/corex_eval/experiments/annotation/bert_finetuned_career/config.yaml",
)

## 8. Free GPU memory

In [ ]:
import gc
model.cpu()
del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory freed.")